# Week 7.2: Bias Mitigation with Activation Steering

In this practical, we'll explore how to measure and mitigate bias in language models
using activation steering - a technique that modifies internal representations
during inference without retraining the model.

## Setup

In [ ]:
# Load CrowS-Pairs dataset from GitHub
crows_url = "https://raw.githubusercontent.com/nyu-mll/crows-pairs/master/data/crows_pairs_anonymized.csv"
crows_df = pd.read_csv(crows_url)

# Filter for gender bias examples only
crows_gender_df = crows_df[crows_df["bias_type"] == "gender"]

# Extract pairs: (stereotypical, anti-stereotypical)
# Note: sent_more is ALWAYS more stereotypical, sent_less is ALWAYS less stereotypical
crows_pairs_gender = []
for _, row in crows_gender_df.iterrows():
    crows_pairs_gender.append((row["sent_more"], row["sent_less"]))

print(f"Loaded {len(crows_pairs_gender)} gender bias pairs from CrowS-Pairs dataset")

In [ ]:
import warnings
import torch
import pandas as pd
from master_mind.teaching.hf import load_hf_model, load_hf_tokenizer


In [ ]:
def get_best_device():
    """Returns the best device on this computer"""

    if torch.cuda.is_available():
        device = torch.device("cuda")
        total_memory = torch.cuda.get_device_properties(device).total_memory
        print(f"GPU Memory: {total_memory / 1e9:.1f} GB")
        print(f"GPU Name: {torch.cuda.get_device_name(device)}")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Found device: {device}")
    return device


device = get_best_device()

In [ ]:
warnings.filterwarnings("ignore")

## Exercise 1: Measuring Bias with CrowS-Pairs

CrowS-Pairs is a benchmark dataset for measuring stereotypical biases in language models.
Each example contains a pair of sentences:
- A **stereotypical** sentence (reinforces a stereotype)
- An **anti-stereotypical** sentence (challenges the stereotype)

We measure bias by checking which sentence the model assigns higher probability to.
A perfectly unbiased model would prefer each sentence 50% of the time.

### CrowS-Pairs Dataset

In [ ]:
print(f"Number of gender bias pairs: {len(crows_pairs_gender)}")
print("\nExample pairs:")
for i, (stereo, anti) in enumerate(crows_pairs_gender[:3]):
    print(f"\nPair {i+1}:")
    print(f"  Stereotypical:     {stereo}")
    print(f"  Anti-stereotypical: {anti}")

In [ ]:
# Load model and tokenizer
tokenizer = load_hf_tokenizer("gpt2", GPT2Tokenizer)
tokenizer.pad_token = tokenizer.eos_token

model = load_hf_model("gpt2", GPT2LMHeadModel).to(device)
model.eval()

In [ ]:
@torch.no_grad()
def compute_sentence_log_prob(sentence: str, model, tokenizer) -> float:
    """Compute log-likelihood of a sentence under the model."""
    inputs = tokenizer(sentence, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]

    outputs = model(**inputs)
    logits = outputs.logits

    # Get log probabilities
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Shift: predict next token from current position
    shift_log_probs = log_probs[0, :-1, :]
    shift_labels = input_ids[0, 1:]

    # Gather log probs for actual tokens
    token_log_probs = shift_log_probs.gather(
        dim=-1, index=shift_labels.unsqueeze(-1)
    ).squeeze(-1)

    return token_log_probs.sum().item()

### Measuring Baseline Bias

In [ ]:
# Implement bias measurement

# Count how often the model prefers the stereotypical sentence
# Bias score = stereotypical_wins / total_pairs
assert False, 'Not implemented yet'


print(f"Baseline bias score: {baseline_bias_score:.2%}")
print("(50% = no bias, >50% = stereotypical bias)")

## Exercise 2: Activation Steering

Activation steering modifies the model's internal representations during inference
to steer its behavior. The key idea:

1. **Find a steering direction**: Compute the difference between activations for
   stereotypical vs anti-stereotypical sentences
2. **Apply steering**: Add/subtract this direction from activations during generation

This allows bias mitigation without retraining!

### Computing Steering Vector

In [ ]:
# Hook to capture activations
class ActivationCapture:
    def __init__(self):
        self.activations = None

    def hook(self, module, input, output):
        # Capture the hidden states (output of transformer block)
        if isinstance(output, tuple):
            self.activations = output[0].detach()
        else:
            self.activations = output.detach()


# We'll steer at a middle layer (layer 6 of GPT-2's 12 layers)
steering_layer = 6

# Compute average activations for stereotypical and anti-stereotypical sentences
stereo_activations = []
anti_activations = []

capture = ActivationCapture()
handle = model.transformer.h[steering_layer].register_forward_hook(capture.hook)


# [[unindent]]
pairs_to_use = crows_pairs_gender
# [[/unindent]]

with torch.no_grad():
    for stereo, anti in pairs_to_use:
        # Get activations for stereotypical sentence
        inputs = tokenizer(stereo, return_tensors="pt").to(device)
        model(**inputs)
        stereo_activations.append(
            capture.activations.mean(dim=1)
        )  # Average over sequence

        # Get activations for anti-stereotypical sentence
        inputs = tokenizer(anti, return_tensors="pt").to(device)
        model(**inputs)
        anti_activations.append(capture.activations.mean(dim=1))

handle.remove()

# Compute steering vector: direction from stereotypical to anti-stereotypical
stereo_mean = torch.stack(stereo_activations).mean(dim=0)
anti_mean = torch.stack(anti_activations).mean(dim=0)
steering_vector = anti_mean - stereo_mean

print(f"Steering vector shape: {steering_vector.shape}")
print(f"Steering vector norm: {steering_vector.norm().item():.4f}")

### Applying Activation Steering

In [ ]:
class SteeringHook:
    def __init__(self, steering_vector, strength=1.0):
        self.steering_vector = steering_vector
        self.strength = strength

    def hook(self, module, input, output):
        if isinstance(output, tuple):
            hidden_states = output[0]
            # Add steering vector to all positions
            steered = hidden_states + self.strength * self.steering_vector
            return (steered,) + output[1:]
        else:
            return output + self.strength * self.steering_vector


# Measure bias with steering applied

# 1. Register the steering hook on the same layer
# 2. Measure bias score with steering active
# 3. Compare to baseline
assert False, 'Not implemented yet'


# Remove the forward hook
handle.remove()

print(f"Baseline bias score: {baseline_bias_score:.2%}")
print(f"Steered bias score:  {steered_bias_score:.2%}")
print(f"Bias reduction:      {baseline_bias_score - steered_bias_score:.2%}")

## Exercise 3: Exploring Steering Strength

The steering strength controls how much we modify the activations.
Let's explore how different strengths affect bias.

### Steering Strength Analysis

In [ ]:
strengths = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
bias_scores = []

for strength in strengths:
    steering_hook = SteeringHook(steering_vector, strength=strength)
    handle = model.transformer.h[steering_layer].register_forward_hook(
        steering_hook.hook
    )

    stereo_wins = 0
    for stereo, anti in crows_pairs_gender:
        prob_stereo = compute_sentence_log_prob(stereo, model, tokenizer)
        prob_anti = compute_sentence_log_prob(anti, model, tokenizer)
        if prob_stereo > prob_anti:
            stereo_wins += 1

    bias_score = stereo_wins / len(crows_pairs_gender)
    bias_scores.append(bias_score)
    handle.remove()

    print(f"Strength {strength:.1f}: Bias score = {bias_score:.2%}")